In [1]:
from ortools.linear_solver import pywraplp

# Производственная задача, симплекс-метод

## Задача 1
- «Шоколадная фабрика» производит 4 вида кондитерских изделий: Kitkat, Twix, шоколад, ириски. Для их производства необходимо затратить определенное количество ингредиентов в граммах.
- В таблице представлены нормативные затраты ресурсов на производство одного изделия
| Заголовок 1 | Шоколад | Печенье | Нуга | Карамель |
|-------------|-------------|-------------|-------------|-------------|
| Kitkat  | 5  | 15  | 5  | 5  |
| Twix  | 10  | 10  | 3  | 10  |
| Шоколад  | 20 | 5 | 0  | 0  |
| Ириска  | 0  | 0  | 3  | 20  |

- На складе завода хранится 30000г. шоколада, 60000г. печенья, 15000г. нуги, 25000г. карамели.
- Необходимо определить сколько штук каждого из видов кондитерских изделий нужно произвести заводу, чтобы получить максимальную прибыль, если цена продажи за единицу товара:
    - Kitkat 30 рублей
    - Twix 40 рублей
    - Шоколад 25 рублей
    - Ириска 15 рублей

Формализуем задачу
- $\large a_i = {30;40;25;15} (m=4)$ - $\large a_i$ - прибыль за штуку изделия;
- $\large b_j = {30000;60000;15000;25000} (n=4)$ - $\large b_j$ - имеющиеся на складе запасы ингредиентов;
- $\large c_ij \begin{bmatrix}
5&15&5&5 \\
10&10&3&10 \\
20&5&0&0 \\
0&0&3&20
\end{bmatrix}$ - количество в г, используемых j-х ингредиентов для i-го изделия.

In [4]:
# ШАГ 1: Создаем решатель.
solver = pywraplp.Solver.CreateSolver('GLOP')

# ШАГ 2: Создаем переменные.
# (0, solver.infinity()) — это условие неотрицательности (x >= 0).
x1 = solver.NumVar(0, solver.infinity(), 'x1')
x2 = solver.NumVar(0, solver.infinity(), 'x2')
x3 = solver.NumVar(0, solver.infinity(), 'x3')
x4 = solver.NumVar(0, solver.infinity(), 'x4')

# ШАГ 3: Добавляем ограничения (ресурсы).
# Ограничение 1: Шоколад (на складе 30 000 г)
constraint_choco = solver.Constraint(-solver.infinity(), 30000, 'chocolate')
constraint_choco.SetCoefficient(x1, 5)
constraint_choco.SetCoefficient(x2, 10)
constraint_choco.SetCoefficient(x3, 20)
constraint_choco.SetCoefficient(x4, 0)

# Ограничение 2: Печенье (на складе 60 000 г)
constraint_biscuit = solver.Constraint(-solver.infinity(), 60000, 'biscuits')
constraint_biscuit.SetCoefficient(x1, 15)
constraint_biscuit.SetCoefficient(x2, 10)
constraint_biscuit.SetCoefficient(x3, 5)
constraint_biscuit.SetCoefficient(x4, 0)

# Ограничение 3: Нуга (на складе 15 000 г)
constraint_nougat = solver.Constraint(-solver.infinity(), 15000, 'nougat')
constraint_nougat.SetCoefficient(x1, 5)
constraint_nougat.SetCoefficient(x2, 3)
constraint_nougat.SetCoefficient(x3, 0)
constraint_nougat.SetCoefficient(x4, 3)

# Ограничение 4: Карамель (на складе 25 000 г)
constraint_caramel = solver.Constraint(-solver.infinity(), 25000, 'caramel')
constraint_caramel.SetCoefficient(x1, 5)
constraint_caramel.SetCoefficient(x2, 10)
constraint_caramel.SetCoefficient(x3, 0)
constraint_caramel.SetCoefficient(x4, 20)

# ШАГ 4: Целевая функция (Прибыль)
objective = solver.Objective()
objective.SetCoefficient(x1, 30)
objective.SetCoefficient(x2, 40)
objective.SetCoefficient(x3, 25)
objective.SetCoefficient(x4, 15)
objective.SetMaximization()

# ШАГ 5: Запуск алгоритма и вывод
solver.Solve()

print("=== Оптимальный план производства ===")
print("Максимальная прибыль:", objective.Value(), "руб.")
print("Произвести Kitkat (x1):", x1.solution_value(), "шт.")
print("Произвести Twix (x2):", x2.solution_value(), "шт.")
print("Произвести Шоколад (x3):", x3.solution_value(), "шт.")
print("Произвести Ириски (x4):", x4.solution_value(), "шт.")


=== Оптимальный план производства ===
Максимальная прибыль: 127678.57142857143 руб.
Произвести Kitkat (x1): 2142.8571428571427 шт.
Произвести Twix (x2): 1428.5714285714287 шт.
Произвести Шоколад (x3): 250.00000000000009 шт.
Произвести Ириски (x4): 0.0 шт.


Много вводных = много кода. Исправим это совершив концептуальный переход от таблички к коду
- Ограничения всегда строятся вокруг ресурсов и в таблице это столбцы и это можно переписать так
    - шоколад: $5x_1 + 10x_2 + 20x_3 + 0x_4 \le 30000 $
    - печенье: $15x_1 + 10x_2 + 5x_3 + 0x_4 \le 60000 $
    - нуга: $5x_1 + 3x_2 + 0x_3 + 3x_4 \le 15000 $
    - карамель: $5x_1 + 10x_2 + 0x_3 + 20x_4 \le 25000 $
- А целевую функцию можно переписать так: $C = 30x_1 + 40x_2 + 25x_3 + 15x_4 \rightarrow \max$


In [27]:
solver = pywraplp.Solver.CreateSolver('GLOP')

# 1. Подготавливаем данные списочно
products = ['Kitkat', 'Twix', 'Шоколад', 'Ириска']
profits = [30, 40, 25, 15]

resources = ['Шоколад', 'Печенье', 'Нуга', 'Карамель']
capacities = [30000, 60000, 15000, 25000]

# Матрица затрат ресурсов.
# Строки здесь — это ресурсы, а столбцы — продукты. "транспонировали" таблицу для удобства обхода.
consumption = [
    [5, 10, 20, 0],   # Затраты шоколада на каждый из 4 продуктов
    [15, 10, 5, 0],   # Затраты печенья
    [5, 3, 0, 3],     # Затраты нуги
    [5, 10, 0, 20]    # Затраты карамели
]

# 2. Создаем переменные в виде списка (x[0] = Kitkat, x[1] = Twix и т.д.)
x = []
for i in range(len(products)):
    x.append(solver.NumVar(0, solver.infinity(), products[i]))

# 3. Добавляем ограничения через циклы
for i in range(len(resources)):
    # Создаем ограничение: от 0 до доступного объема на складе
    constraint = solver.Constraint(-solver.infinity(), capacities[i], resources[i])

    # Заполняем коэффициенты для этого ограничения
    for j in range(len(products)):
        constraint.SetCoefficient(x[j], consumption[i][j])

# 4. Целевая функция
objective = solver.Objective()
for j in range(len(products)):
    objective.SetCoefficient(x[j], profits[j])
    objective.SetMaximization()

# 5. Решаем
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("=== Оптимальный план производства ===")
    print(f"Максимальная прибыль: {objective.Value()} руб.\n")

    for j in range(len(products)):
    # Выводим только те позиции, которые реально стоит производить
    # Используем округление, чтобы избежать артефактов вроде 0.00000000001
        amount = round(x[j].solution_value(), 2)
        print(f"Произвести {products[j]}: {amount} шт.")
else:
   print("Оптимальное решение не найдено.")

=== Оптимальный план производства ===
Максимальная прибыль: 127678.57142857143 руб.

Произвести Kitkat: 2142.86 шт.
Произвести Twix: 1428.57 шт.
Произвести Шоколад: 250.0 шт.
Произвести Ириска: 0.0 шт.
